# Combined Qualitative Figures from Tar Archives

This notebook rebuilds both publication-ready qualitative figures from the three staged tar archives:

- `vsb7_3600_rare_first_portable.tar`
- `vsb7_3600_rare_first_yolo.tar`
- `vnwoodknot_live_dead_2class_yolo.tar`

It covers two outputs in one run:

1. **In-domain curated benchmark figure**
   - Ground truth
   - Faster R-CNN lightweight reference
   - YOLOv8s baseline (`Y0-e200`)
   - YOLO P2 variant (`Y1-e200`)

2. **VNWoodKnot target-domain figure**
   - Ground truth
   - `T0`: target-only training
   - `T1`: source-initialized fine-tuning from `Y0-e200`

Why this notebook is tar-based:
- the heavy benchmark-building step is done locally
- Colab only extracts the prepared subsets
- YOLO training reads from local `/content`, not directly from Drive
- final artifacts are written back to Drive

Notes:
- This notebook assumes the three tar files have already been uploaded to the project Drive folder.
- It uses `Y1` as the in-domain variant because it is branch-free and easier to rerun reliably on Colab.
- The VNWoodKnot manifest is rebuilt from the extracted YOLO dataset so the existing evaluator and figure builder can be reused cleanly.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


Mounted at /content/drive
/bin/bash: line 1: nvidia-smi: command not found


In [2]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DRIVE_ROOT = Path('/content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2')
ARCHIVE_ROOT = PROJECT_DRIVE_ROOT / 'data'
REPO_URL = 'https://github.com/khanhnt/wood-defect-q2'
REPO_DIR = Path('/content/wood-defect-q2')

MAIN_PORTABLE_ROOT = Path('/content/main_dataset/benchmarks/vsb7_3600_rare_first_portable')
MAIN_YOLO_ROOT = Path('/content/main_dataset/benchmarks/vsb7_3600_rare_first_yolo')
VN_YOLO_ROOT = Path('/content/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo')

ARTIFACT_ROOT = PROJECT_DRIVE_ROOT / 'rerun_artifacts'
IN_DOMAIN_ROOT = ARTIFACT_ROOT / 'in_domain'
VN_TRANSFER_ROOT = ARTIFACT_ROOT / 'vnwoodknot'

IMAGE_SIZE = 1024
TWO_STAGE_EPOCHS = 10
IN_DOMAIN_YOLO_EPOCHS = 200
VN_EPOCHS = 50
IN_DOMAIN_ROWS = 5
VN_ROWS = 4

RUN_TWO_STAGE = True
RUN_IN_DOMAIN_YOLO = True
RUN_VN_TRANSFER = True

MAIN_CLASSES = [
    'live_knot',
    'dead_knot',
    'resin',
    'knot_with_crack',
    'crack',
    'marrow',
    'knot_missing',
]
VN_CLASSES = ['live_knot', 'dead_knot']

for root in [ARTIFACT_ROOT, IN_DOMAIN_ROOT, VN_TRANSFER_ROOT]:
    root.mkdir(parents=True, exist_ok=True)

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['WOOD_MAIN_PROCESSED_ROOT'] = str(MAIN_PORTABLE_ROOT)
os.environ['WOOD_VN_PROCESSED_ROOT'] = str(VN_YOLO_ROOT)


def run(cmd, cwd=REPO_DIR):
    printable = ' '.join(str(part) for part in cmd)
    print(f'$ {printable}')
    subprocess.run([str(part) for part in cmd], cwd=str(cwd), check=True)


def assert_exists(path: Path):
    if not path.exists():
        raise FileNotFoundError(path)
    return path

for path in [
    ARCHIVE_ROOT / 'vsb7_3600_rare_first_portable.tar',
    ARCHIVE_ROOT / 'vsb7_3600_rare_first_yolo.tar',
    ARCHIVE_ROOT / 'vnwoodknot_live_dead_2class_yolo.tar',
]:
    print(path, 'OK' if path.exists() else 'MISSING')

print('ARTIFACT_ROOT =', ARTIFACT_ROOT)
print('IN_DOMAIN_ROOT =', IN_DOMAIN_ROOT)
print('VN_TRANSFER_ROOT =', VN_TRANSFER_ROOT)


/content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/vsb7_3600_rare_first_portable.tar OK
/content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/vsb7_3600_rare_first_yolo.tar OK
/content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/vnwoodknot_live_dead_2class_yolo.tar OK
ARTIFACT_ROOT = /content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/rerun_artifacts
IN_DOMAIN_ROOT = /content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/rerun_artifacts/in_domain
VN_TRANSFER_ROOT = /content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/rerun_artifacts/vnwoodknot


In [3]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run(['git', 'clone', REPO_URL, str(REPO_DIR)], cwd='/content')
run([
    'python3', '-m', 'pip', 'install', '-q',
    'ultralytics==8.3.0',
    'timm',
    'pycocotools',
    'pandas',
    'pillow',
    'pyyaml',
], cwd='/content')


$ git clone https://github.com/khanhnt/wood-defect-q2 /content/wood-defect-q2
$ python3 -m pip install -q ultralytics==8.3.0 timm pycocotools pandas pillow pyyaml


In [4]:
for stale_path in [Path('/content/main_dataset'), Path('/content/vnwoodknot')]:
    if stale_path.exists():
        shutil.rmtree(stale_path)

for archive_name in [
    'vsb7_3600_rare_first_portable.tar',
    'vsb7_3600_rare_first_yolo.tar',
    'vnwoodknot_live_dead_2class_yolo.tar',
]:
    run(['tar', '-xf', str(ARCHIVE_ROOT / archive_name), '-C', '/content'], cwd='/content')

for path in [
    MAIN_PORTABLE_ROOT / 'manifest.jsonl',
    MAIN_YOLO_ROOT / 'dataset.yaml',
    VN_YOLO_ROOT / 'dataset.yaml',
]:
    print(path, 'OK' if path.exists() else 'MISSING')


$ tar -xf /content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/vsb7_3600_rare_first_portable.tar -C /content
$ tar -xf /content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/vsb7_3600_rare_first_yolo.tar -C /content
$ tar -xf /content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/vnwoodknot_live_dead_2class_yolo.tar -C /content
/content/main_dataset/benchmarks/vsb7_3600_rare_first_portable/manifest.jsonl OK
/content/main_dataset/benchmarks/vsb7_3600_rare_first_yolo/dataset.yaml OK
/content/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/dataset.yaml OK


In [5]:
import json
from textwrap import dedent

import yaml
from PIL import Image

GENERATED_DIR = REPO_DIR / 'configs' / 'generated_colab'
MODELS_DIR = REPO_DIR / 'configs' / 'models'
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_MAIN_CFG_PATH = GENERATED_DIR / 'dataset_main_vsb7_3600_rarefirst_portable.yaml'
DATASET_VN_CFG_PATH = GENERATED_DIR / 'dataset_vnwoodknot_from_yolo.yaml'
BASELINE_TRAIN_CFG_PATH = GENERATED_DIR / 'train_baseline_vsb7_3600_rarefirst_full.yaml'
BASELINE_EVAL_CFG_PATH = GENERATED_DIR / 'eval_baseline_vsb7_3600_rarefirst_full.yaml'
Y1_MODEL_PATH = MODELS_DIR / 'yolov8s-p2-7class.yaml'
VN_MANIFEST_PATH = VN_YOLO_ROOT / 'manifest.jsonl'
VN_METADATA_PATH = VN_YOLO_ROOT / 'metadata_rebuilt.json'


def write_yaml(path: Path, payload: dict):
    path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')


def rewrite_yolo_dataset_yaml(dataset_root: Path, class_names: list[str]):
    dataset_yaml = dataset_root / 'dataset.yaml'
    payload = {
        'path': str(dataset_root),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {index: name for index, name in enumerate(class_names)},
    }
    write_yaml(dataset_yaml, payload)
    return dataset_yaml


def build_vn_manifest_from_yolo(dataset_root: Path, class_names: list[str]):
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    records = []
    split_counts = {}
    annotation_count = 0

    for split in ['train', 'val', 'test']:
        image_dir = dataset_root / 'images' / split
        label_dir = dataset_root / 'labels' / split
        split_count = 0
        for image_path in sorted(image_dir.iterdir()):
            if image_path.suffix.lower() not in image_exts:
                continue
            with Image.open(image_path) as image:
                width, height = image.size
            label_path = label_dir / f'{image_path.stem}.txt'
            annotations = []
            if label_path.exists():
                lines = label_path.read_text(encoding='utf-8').splitlines()
                for line_index, line in enumerate(lines):
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    class_id = int(float(parts[0]))
                    cx = float(parts[1])
                    cy = float(parts[2])
                    box_w = float(parts[3])
                    box_h = float(parts[4])
                    x1 = max(0.0, cx - box_w * 0.5)
                    y1 = max(0.0, cy - box_h * 0.5)
                    x2 = min(1.0, cx + box_w * 0.5)
                    y2 = min(1.0, cy + box_h * 0.5)
                    annotations.append({
                        'annotation_id': f'{image_path.stem}_{line_index}',
                        'class_id': class_id,
                        'class_name': class_names[class_id],
                        'bbox_xyxy_norm': [x1, y1, x2, y2],
                        'bbox_width_norm': box_w,
                        'bbox_height_norm': box_h,
                        'bbox_area_norm': box_w * box_h,
                    })
            annotation_count += len(annotations)
            records.append({
                'image_id': image_path.stem,
                'image_path': str(Path('images') / split / image_path.name),
                'width': width,
                'height': height,
                'split': split,
                'annotations': annotations,
            })
            split_count += 1
        split_counts[split] = split_count

    with VN_MANIFEST_PATH.open('w', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record) + '\\n')

    VN_METADATA_PATH.write_text(
        json.dumps(
            {
                'dataset_name': 'vnwoodknot_live_dead_2class_yolo_rebuilt',
                'root_dir': str(dataset_root),
                'classes': class_names,
                'num_records_by_split': split_counts,
                'num_annotations': annotation_count,
            },
            indent=2,
        ),
        encoding='utf-8',
    )
    return split_counts, annotation_count


rewrite_yolo_dataset_yaml(MAIN_YOLO_ROOT, MAIN_CLASSES)
rewrite_yolo_dataset_yaml(VN_YOLO_ROOT, VN_CLASSES)

main_dataset_cfg = {
    'dataset_name': 'large_scale_wood_surface_defects_vsb7_3600_rare_first_portable',
    'root_dir': str(MAIN_PORTABLE_ROOT),
    'manifest_path': str(MAIN_PORTABLE_ROOT / 'manifest.jsonl'),
    'classes': MAIN_CLASSES,
    'small_defect': {
        'enabled': True,
        'combine': 'any',
        'min_area_ratio': 0.01,
        'min_width_px': 16,
        'min_height_px': 16,
    },
}
write_yaml(DATASET_MAIN_CFG_PATH, main_dataset_cfg)

vn_split_counts, vn_annotation_count = build_vn_manifest_from_yolo(VN_YOLO_ROOT, VN_CLASSES)

vn_dataset_cfg = {
    'dataset_name': 'vnwoodknot_live_dead_2class_yolo_rebuilt',
    'root_dir': str(VN_YOLO_ROOT),
    'manifest_path': str(VN_MANIFEST_PATH),
    'classes': VN_CLASSES,
}
write_yaml(DATASET_VN_CFG_PATH, vn_dataset_cfg)

baseline_train_cfg = {
    'seed': 42,
    'device': 'cuda',
    'output_dir': str(IN_DOMAIN_ROOT),
    'experiment_name': 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full',
    'dataset': {
        'train': 'configs/generated_colab/dataset_main_vsb7_3600_rarefirst_portable.yaml',
        'val': 'configs/generated_colab/dataset_main_vsb7_3600_rarefirst_portable.yaml',
        'train_split': 'train',
        'val_split': 'val',
    },
    'dataset_split': {
        'seed': 42,
        'train_ratio': 0.8,
        'val_ratio': 0.1,
    },
    'train': {
        'epochs': int(TWO_STAGE_EPOCHS),
        'batch_size': 2,
        'num_workers': 2,
        'image_size': int(IMAGE_SIZE),
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        'best_metric': 'mAP50_95',
        'small_defect_sampler': {
            'enabled': False,
            'small_weight': 3.0,
            'positive_weight': 1.5,
            'negative_weight': 0.5,
        },
    },
    'model': {
        'name': 'baseline_detector',
        'num_classes': 7,
        'backbone': 'mobilenet_hr',
        'image_size': int(IMAGE_SIZE),
        'score_threshold': 0.05,
        'nms_threshold': 0.5,
        'max_detections': 100,
    },
}
write_yaml(BASELINE_TRAIN_CFG_PATH, baseline_train_cfg)

baseline_eval_cfg = {
    'seed': 42,
    'device': 'cuda',
    'output_dir': str(IN_DOMAIN_ROOT),
    'experiment_name': 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full_eval',
    'checkpoint_path': str(IN_DOMAIN_ROOT / 'checkpoints' / 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full' / 'best.pt'),
    'dataset': {
        'eval': 'configs/generated_colab/dataset_main_vsb7_3600_rarefirst_portable.yaml',
        'split': 'test',
    },
    'dataset_split': {
        'seed': 42,
        'train_ratio': 0.8,
        'val_ratio': 0.1,
    },
    'model': {
        'name': 'baseline_detector',
        'num_classes': 7,
        'backbone': 'mobilenet_hr',
        'image_size': int(IMAGE_SIZE),
        'score_threshold': 0.05,
        'nms_threshold': 0.5,
        'max_detections': 100,
    },
    'evaluation': {
        'batch_size': 1,
        'num_workers': 2,
        'score_threshold': 0.05,
        'compute_small_defect_eval': True,
        'tile_merge': False,
        'save_predictions': True,
        'save_visualizations': False,
        'compute_per_class_ap': True,
        'compute_cross_dataset': False,
    },
}
write_yaml(BASELINE_EVAL_CFG_PATH, baseline_eval_cfg)

y1_model_yaml = dedent('''# Ultralytics YOLOv8s-P2 detection model specialized for the 7-class wood-defect setup.
nc: 7
depth_multiple: 0.33
width_multiple: 0.50

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[18, 21, 24, 27], 1, Detect, [nc]]
''')
Y1_MODEL_PATH.write_text(y1_model_yaml, encoding='utf-8')

print('Generated:')
for path in [
    DATASET_MAIN_CFG_PATH,
    DATASET_VN_CFG_PATH,
    BASELINE_TRAIN_CFG_PATH,
    BASELINE_EVAL_CFG_PATH,
    Y1_MODEL_PATH,
    VN_MANIFEST_PATH,
    VN_METADATA_PATH,
]:
    print(' -', path)
print('VN split counts:', vn_split_counts)
print('VN annotations:', vn_annotation_count)


Generated:
 - /content/wood-defect-q2/configs/generated_colab/dataset_main_vsb7_3600_rarefirst_portable.yaml
 - /content/wood-defect-q2/configs/generated_colab/dataset_vnwoodknot_from_yolo.yaml
 - /content/wood-defect-q2/configs/generated_colab/train_baseline_vsb7_3600_rarefirst_full.yaml
 - /content/wood-defect-q2/configs/generated_colab/eval_baseline_vsb7_3600_rarefirst_full.yaml
 - /content/wood-defect-q2/configs/models/yolov8s-p2-7class.yaml
 - /content/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/manifest.jsonl
 - /content/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/metadata_rebuilt.json
VN split counts: {'train': 1060, 'val': 226, 'test': 229}
VN annotations: 1021


In [6]:
for path in [
    MAIN_PORTABLE_ROOT / 'manifest.jsonl',
    MAIN_YOLO_ROOT / 'dataset.yaml',
    VN_YOLO_ROOT / 'dataset.yaml',
    VN_MANIFEST_PATH,
    DATASET_MAIN_CFG_PATH,
    DATASET_VN_CFG_PATH,
]:
    print(path, 'OK' if path.exists() else 'MISSING')


/content/main_dataset/benchmarks/vsb7_3600_rare_first_portable/manifest.jsonl OK
/content/main_dataset/benchmarks/vsb7_3600_rare_first_yolo/dataset.yaml OK
/content/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/dataset.yaml OK
/content/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/manifest.jsonl OK
/content/wood-defect-q2/configs/generated_colab/dataset_main_vsb7_3600_rarefirst_portable.yaml OK
/content/wood-defect-q2/configs/generated_colab/dataset_vnwoodknot_from_yolo.yaml OK


## In-domain rerun and figure build


In [7]:
if RUN_TWO_STAGE:
    run([
        'python3', 'scripts/train.py',
        '--config', str(BASELINE_TRAIN_CFG_PATH),
        '--experiment-name', 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full',
        '--epochs', str(TWO_STAGE_EPOCHS),
        '--device', 'cuda',
    ])
    run([
        'python3', 'scripts/evaluate.py',
        '--config', str(BASELINE_EVAL_CFG_PATH),
        '--checkpoint', str(IN_DOMAIN_ROOT / 'checkpoints' / 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full' / 'best.pt'),
        '--experiment-name', 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full_eval',
        '--device', 'cuda',
    ])
else:
    print('Skipping two-stage reference training/evaluation.')


$ python3 scripts/train.py --config /content/wood-defect-q2/configs/generated_colab/train_baseline_vsb7_3600_rarefirst_full.yaml --experiment-name baseline_mobilenet_hr_vsb7_3600_rarefirst_full --epochs 10 --device cuda


KeyboardInterrupt: 

In [ ]:
if RUN_IN_DOMAIN_YOLO:
    run([
        'python3', 'scripts/train_yolov8.py',
        '--data', str(MAIN_YOLO_ROOT / 'dataset.yaml'),
        '--model', 'yolov8s',
        '--experiment-name', 'y0_yolov8s_vsb7_3600_rarefirst_e200',
        '--epochs', str(IN_DOMAIN_YOLO_EPOCHS),
        '--imgsz', str(IMAGE_SIZE),
        '--batch', '32',
        '--device', '0',
        '--workers', '4',
        '--seed', '42',
        '--patience', '50',
        '--project-dir', str(IN_DOMAIN_ROOT / 'yolo'),
    ])
    run([
        'python3', 'scripts/evaluate_yolov8.py',
        '--dataset-config', str(DATASET_MAIN_CFG_PATH),
        '--checkpoint', str(IN_DOMAIN_ROOT / 'yolo' / 'y0_yolov8s_vsb7_3600_rarefirst_e200' / 'weights' / 'best.pt'),
        '--experiment-name', 'y0_yolov8s_vsb7_3600_rarefirst_e200_eval',
        '--split', 'test',
        '--batch', '8',
        '--imgsz', str(IMAGE_SIZE),
        '--device', '0',
        '--output-dir', str(IN_DOMAIN_ROOT),
        '--small-defect-eval',
        '--save-predictions',
    ])

    run([
        'python3', 'scripts/train_yolov8.py',
        '--data', str(MAIN_YOLO_ROOT / 'dataset.yaml'),
        '--model', str(Y1_MODEL_PATH),
        '--experiment-name', 'y1_yolov8s_p2_vsb7_3600_rarefirst_e200',
        '--epochs', str(IN_DOMAIN_YOLO_EPOCHS),
        '--imgsz', str(IMAGE_SIZE),
        '--batch', '16',
        '--device', '0',
        '--workers', '4',
        '--seed', '42',
        '--patience', '50',
        '--project-dir', str(IN_DOMAIN_ROOT / 'yolo'),
    ])
    run([
        'python3', 'scripts/evaluate_yolov8.py',
        '--dataset-config', str(DATASET_MAIN_CFG_PATH),
        '--checkpoint', str(IN_DOMAIN_ROOT / 'yolo' / 'y1_yolov8s_p2_vsb7_3600_rarefirst_e200' / 'weights' / 'best.pt'),
        '--experiment-name', 'y1_yolov8s_p2_vsb7_3600_rarefirst_e200_eval',
        '--split', 'test',
        '--batch', '8',
        '--imgsz', str(IMAGE_SIZE),
        '--device', '0',
        '--output-dir', str(IN_DOMAIN_ROOT),
        '--small-defect-eval',
        '--save-predictions',
    ])
else:
    print('Skipping in-domain YOLO reruns.')


In [ ]:
baseline_predictions = IN_DOMAIN_ROOT / 'tables' / 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full_eval_test_predictions.jsonl'
y0_predictions = IN_DOMAIN_ROOT / 'tables' / 'y0_yolov8s_vsb7_3600_rarefirst_e200_eval_test_predictions.jsonl'
y1_predictions = IN_DOMAIN_ROOT / 'tables' / 'y1_yolov8s_p2_vsb7_3600_rarefirst_e200_eval_test_predictions.jsonl'

for path in [baseline_predictions, y0_predictions, y1_predictions]:
    assert_exists(path)

run([
    'python3', 'scripts/build_in_domain_qualitative_figure.py',
    '--manifest', str(MAIN_PORTABLE_ROOT / 'manifest.jsonl'),
    '--image-root-dir', str(MAIN_PORTABLE_ROOT),
    '--split', 'test',
    '--rows', str(IN_DOMAIN_ROWS),
    '--baseline-run-name', 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full',
    '--baseline-header', 'Faster R-CNN',
    '--baseline-predictions', str(baseline_predictions),
    '--yolo-run-name', 'y0_yolov8s_vsb7_3600_rarefirst_e200',
    '--yolo-header', 'YOLOv8s',
    '--yolo-predictions', str(y0_predictions),
    '--variant-run-name', 'y1_yolov8s_p2_vsb7_3600_rarefirst_e200',
    '--variant-header', 'YOLO P2',
    '--variant-predictions', str(y1_predictions),
    '--output-dir', str(IN_DOMAIN_ROOT / 'figures' / 'in_domain_qualitative'),
])


## VNWoodKnot transfer rerun and figure build


In [ ]:
if RUN_VN_TRANSFER:
    y0_checkpoint = IN_DOMAIN_ROOT / 'yolo' / 'y0_yolov8s_vsb7_3600_rarefirst_e200' / 'weights' / 'best.pt'
    assert_exists(y0_checkpoint)

    run([
        'python3', 'scripts/train_yolov8.py',
        '--data', str(VN_YOLO_ROOT / 'dataset.yaml'),
        '--model', 'yolov8s',
        '--experiment-name', 't0_yolov8s_vnwoodknot_target_only_e50',
        '--epochs', str(VN_EPOCHS),
        '--imgsz', str(IMAGE_SIZE),
        '--batch', '32',
        '--device', '0',
        '--workers', '4',
        '--seed', '42',
        '--patience', '30',
        '--project-dir', str(VN_TRANSFER_ROOT / 'yolo'),
    ])
    run([
        'python3', 'scripts/train_yolov8.py',
        '--data', str(VN_YOLO_ROOT / 'dataset.yaml'),
        '--weights', str(y0_checkpoint),
        '--experiment-name', 't1_y0_3600e200_to_vnwoodknot_e50',
        '--epochs', str(VN_EPOCHS),
        '--imgsz', str(IMAGE_SIZE),
        '--batch', '32',
        '--device', '0',
        '--workers', '4',
        '--seed', '42',
        '--patience', '30',
        '--project-dir', str(VN_TRANSFER_ROOT / 'yolo'),
    ])
else:
    print('Skipping VNWoodKnot transfer reruns.')


In [ ]:
t0_checkpoint = VN_TRANSFER_ROOT / 'yolo' / 't0_yolov8s_vnwoodknot_target_only_e50' / 'weights' / 'best.pt'
t1_checkpoint = VN_TRANSFER_ROOT / 'yolo' / 't1_y0_3600e200_to_vnwoodknot_e50' / 'weights' / 'best.pt'
for path in [t0_checkpoint, t1_checkpoint]:
    assert_exists(path)

run([
    'python3', 'scripts/evaluate_yolov8.py',
    '--dataset-config', str(DATASET_VN_CFG_PATH),
    '--checkpoint', str(t0_checkpoint),
    '--experiment-name', 't0_yolov8s_vnwoodknot_target_only_e50_eval',
    '--split', 'test',
    '--batch', '8',
    '--imgsz', str(IMAGE_SIZE),
    '--device', '0',
    '--output-dir', str(VN_TRANSFER_ROOT),
    '--save-predictions',
])
run([
    'python3', 'scripts/evaluate_yolov8.py',
    '--dataset-config', str(DATASET_VN_CFG_PATH),
    '--checkpoint', str(t1_checkpoint),
    '--experiment-name', 't1_y0_3600e200_to_vnwoodknot_e50_eval',
    '--split', 'test',
    '--batch', '8',
    '--imgsz', str(IMAGE_SIZE),
    '--device', '0',
    '--output-dir', str(VN_TRANSFER_ROOT),
    '--save-predictions',
])


In [ ]:
t0_predictions = VN_TRANSFER_ROOT / 'tables' / 't0_yolov8s_vnwoodknot_target_only_e50_eval_test_predictions.jsonl'
t1_predictions = VN_TRANSFER_ROOT / 'tables' / 't1_y0_3600e200_to_vnwoodknot_e50_eval_test_predictions.jsonl'
for path in [VN_MANIFEST_PATH, t0_predictions, t1_predictions]:
    assert_exists(path)

run([
    'python3', 'scripts/build_vnwoodknot_qualitative_from_predictions.py',
    '--manifest', str(VN_MANIFEST_PATH),
    '--image-root-dir', str(VN_YOLO_ROOT),
    '--split', 'test',
    '--rows', str(VN_ROWS),
    '--class-names', 'live_knot', 'dead_knot',
    '--t0-run-name', 't0_yolov8s_vnwoodknot_target_only_e50',
    '--t0-header', 'T0',
    '--t0-predictions', str(t0_predictions),
    '--t1-run-name', 't1_y0_3600e200_to_vnwoodknot_e50',
    '--t1-header', 'T1',
    '--t1-predictions', str(t1_predictions),
    '--output-dir', str(VN_TRANSFER_ROOT / 'figures' / 'vnwoodknot_transfer_qualitative_raw'),
])


In [ ]:
print('In-domain figure outputs:')
for path in sorted((IN_DOMAIN_ROOT / 'figures' / 'in_domain_qualitative').glob('*')):
    print(' -', path)

print('\nVNWoodKnot figure outputs:')
for path in sorted((VN_TRANSFER_ROOT / 'figures' / 'vnwoodknot_transfer_qualitative_raw').glob('*')):
    print(' -', path)
